In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2025-12-06 17:40:18.018379: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-06 17:40:18.021590: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
import pandas as pd
import numpy as np

In [3]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=2,#cores per slurm job
        memory="32G",#memory per slurm job
        processes=1,#dask workers per slurm jobTrueT
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=3:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

2025-12-06 17:40:20,732 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-06 17:40:20,733 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-06 17:40:20,734 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB
2025-12-06 17:40:20,735 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 48G due to system memory limit of 32.00 GiB


In [4]:
client.dashboard_link

'http://127.0.0.1:8787/status'

In [5]:
DATA_ROOT="/home/mcn26/project_pi_skr2/shared/tabula_data"
simpath="simulated/shendure_pow_analysis/sim_with_orthos_20251203"
scmpraobj=scm.scMPRA_data.from_parquet(f"{DATA_ROOT}/{simpath}/scMPRA/0.scmpra")
dat=scmpraobj.data
o0=scm.ortho.load(client,f"{DATA_ROOT}/{simpath}/orthos","0")
mats=o0.by_cell_type_design["Cardiomyocytes"].result()

manually calc reference beta...

In [6]:
refmean=dat[(dat["cell_type"]=="Cardiomyocytes") & (dat["cre_id"]=="reference")]["umis_mpra_bc"].mean()

In [7]:
refmean

0.03777335984095427

$$e^{\beta_{ref}}=E[ref]\tag{2}$$

In [8]:
np.log(refmean)

-3.2761511909332985

testing new standard fit with MoM

In [9]:
#a,b=scm.standard_fit(client,data=scmpraobj,split="cre_id")

In [10]:
c,d=scm.standard_fit(client,data=scmpraobj,split="cell_type")

In [14]:
c.model["Cardiomyocytes"].result()

{'llf_total': -187714.57378529012,
 'llfs': array([-187714.57378529]),
 'aic_total': 375645.14757058024,
 'aics': array([375645.14757058]),
 'df_model_total': 108,
 'df': 108,
 'weights': {'x_mu': array([[-3.1385934 ],
         [ 8.219519  ],
         [ 7.5197196 ],
         [ 6.2855835 ],
         [ 7.725921  ],
         [ 8.659286  ],
         [ 8.8575115 ],
         [ 8.66504   ],
         [ 7.284152  ],
         [ 8.944525  ],
         [ 6.6893296 ],
         [ 8.415082  ],
         [ 6.3847384 ],
         [ 8.485084  ],
         [ 8.477223  ],
         [ 6.2517133 ],
         [ 7.1328936 ],
         [ 7.970729  ],
         [ 8.434219  ],
         [ 9.247926  ],
         [ 6.5492196 ],
         [ 7.576912  ],
         [ 8.968116  ],
         [ 8.647739  ],
         [ 8.640311  ],
         [ 8.813572  ],
         [ 8.652365  ],
         [ 8.81947   ],
         [ 8.568843  ],
         [ 8.876106  ],
         [ 8.672201  ],
         [ 6.239376  ],
         [ 4.8526273 ],
         [ 7.

In [15]:
client.close()
cluster.close()

2025-12-06 17:44:59,220 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('_tensorzinb_fit-aa75ae13da0438fc43f9aa100e19e5a9')" coro=<Worker.execute() done, defined at /home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-06 17:44:59,220 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('_tensorzinb_fit-3cc5d2080d7df8e2cf3e571421415c81')" coro=<Worker.execute() done, defined at /home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-06 17:44:59,223 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('_tensorzinb_fit-af77bc18e3592b6dceb51192217284fb')" coro=<Worker.execute() done, defined at /home/mcn26/.conda/envs/env_tzinb/lib/python3.10/site-packages/distributed/worker_state_